# Dataset Validation & Analysis

**Purpose**: Validate dataset structure and provide detailed statistics.

**Prerequisites**: Run `01_dataset_pipeline_local.ipynb` first


## Section 1: Imports & Load Data


In [ ]:
import os
import json
import random
from pathlib import Path
from collections import Counter

In [ ]:
# Configure paths
BASE_DIR = Path("/Users/anas/Projects/code-security-identifier")
DATASETS_DIR = BASE_DIR / "datasets"

def read_jsonl(path):
    """
    Read JSONL file into list of records.
    """
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]

# Load datasets
train = read_jsonl(DATASETS_DIR / "FINAL_train.jsonl")
val = read_jsonl(DATASETS_DIR / "FINAL_val.jsonl")

print(f"✓ Train: {len(train):,} functions")
print(f"✓ Val:   {len(val):,} functions")

## Section 2: Structure Validation


In [ ]:
print("Validating dataset structure...\n")

issues = []

for split_name, data in [("train", train), ("val", val)]:
    for i, r in enumerate(data):
        # Check required fields
        required = ["lines", "raw_lines", "label", "type", "cwe_id", "dataset_source"]
        for field in required:
            if field not in r:
                issues.append(f"{split_name}[{i}]: missing '{field}'")

        # Check mismatches
        n_labels = len(r.get("label", []))
        n_lines = len(r.get("lines", []))
        n_raw = len(r.get("raw_lines", []))
        n_types = len(r.get("type", []))

        if n_labels != n_lines:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != lines({n_lines})")
        if n_labels != n_types:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != type({n_types})")
        if n_labels != n_raw:
            issues.append(f"{split_name}[{i}]: label({n_labels}) != raw_lines({n_raw})")

        # Check for empty labels
        if n_labels == 0:
            issues.append(f"{split_name}[{i}]: empty label list")

if issues:
    print(f"⚠ Found {len(issues)} structure issues:")
    for issue in issues[:10]:
        print(f"  - {issue}")
    if len(issues) > 10:
        print(f"  ... and {len(issues) - 10} more")
else:
    print("✓ All datasets have valid structure")
    print(f"✓ All {len(train) + len(val):,} records have correct format")

## Section 3: Distribution Summary


In [ ]:
print("\n" + "="*70)
print("DATASET SUMMARY")
print("="*70)

# Overview
print(f"\nSplit sizes (functions):")
print(f"  Train: {len(train):,}")
print(f"  Val:   {len(val):,}")
print(f"  Total: {len(train) + len(val):,}")

# Vulnerability distribution (functions)
print(f"\nVulnerability distribution (functions):")
for name, data in [("Train", train), ("Val", val)]:
    total = len(data)
    vuln = sum(1 for r in data if sum(r["label"]) > 0)
    safe = total - vuln
    pct = vuln / total * 100 if total else 0
    print(f"  {name:>5}: {vuln:>7,} vulnerable / {safe:>7,} safe ({pct:>5.1f}% vulnerable)")

# Statement-level distribution
print(f"\nVulnerability distribution (statements):")
for name, data in [("Train", train), ("Val", val)]:
    total_s = sum(len(r["label"]) for r in data)
    vuln_s = sum(sum(r["label"]) for r in data)
    safe_s = total_s - vuln_s
    pct = vuln_s / total_s * 100 if total_s else 0
    print(f"  {name:>5}: {vuln_s:>9,} vulnerable / {safe_s:>9,} safe ({pct:>5.1f}% vulnerable)")

## Section 4: Dataset Sources


In [ ]:
print(f"\nDataset sources in training set:")
sources = Counter(r.get("dataset_source", "?") for r in train)
for source, count in sources.most_common():
    pct = count / len(train) * 100
    print(f"  {source:>15}: {count:>7,} ({pct:>5.1f}%)")

print(f"\nDataset sources in validation set:")
sources_val = Counter(r.get("dataset_source", "?") for r in val)
for source, count in sources_val.most_common():
    pct = count / len(val) * 100
    print(f"  {source:>15}: {count:>7,} ({pct:>5.1f}%)")

## Section 5: CWE Distribution


In [ ]:
print(f"\nCWE types in training set (vulnerable functions only):")
cwes = Counter(r.get("cwe_id", "?") for r in train if sum(r["label"]) > 0)
for cwe, count in cwes.most_common():
    print(f"  {cwe:>12}: {count:>6,}")

print(f"\nCWE types in validation set (vulnerable functions only):")
cwes_val = Counter(r.get("cwe_id", "?") for r in val if sum(r["label"]) > 0)
for cwe, count in cwes_val.most_common():
    print(f"  {cwe:>12}: {count:>6,}")

## Section 6: File Sizes


In [ ]:
print(f"\nFile sizes:")
for fname in ["FINAL_train.jsonl", "FINAL_val.jsonl"]:
    path = DATASETS_DIR / fname
    size_mb = os.path.getsize(path) / (1024 * 1024)
    size_lines = sum(1 for _ in open(path))
    print(f"  {fname:>20}: {size_mb:>7.1f} MB ({size_lines:>8,} lines)")

print(f"\n✓ Output directory: {DATASETS_DIR}")

## Section 7: Sample Records


In [ ]:
print("\nRandom vulnerable samples from training set:\n")

# Pick 3 vulnerable functions at random
random.seed(0)
vuln_samples = [r for r in train if sum(r["label"]) > 0]
picks = random.sample(vuln_samples, min(3, len(vuln_samples)))

for i, sample in enumerate(picks):
    cwe = sample.get("cwe_id", "?")
    src = sample.get("dataset_source", "?")
    print(f"--- Sample {i+1} | {cwe} | from: {src} ---")

    lines = sample.get("raw_lines", sample.get("lines", []))
    labels = sample["label"]

    for j, (line, lbl) in enumerate(zip(lines[:15], labels[:15])):
        mark = ">>" if lbl == 1 else "  "
        print(f"  {mark} {j+1:>3} | {line[:75]}")

    if len(lines) > 15:
        print(f"       ... ({len(lines) - 15} more lines)")
    print()

## Section 8: Function Length Distribution


In [ ]:
print("Train split - statements per function:")
lengths = [len(r["label"]) for r in train]
print(f"  Min:    {min(lengths):>6}")
print(f"  Max:    {max(lengths):>6}")
print(f"  Mean:   {sum(lengths)/len(lengths):>6.1f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]:>6}")

# Percentiles
over_50 = sum(1 for l in lengths if l > 50)
over_100 = sum(1 for l in lengths if l > 100)
over_200 = sum(1 for l in lengths if l > 200)
print(f"\n  >50 statements:   {over_50:>6,} ({over_50/len(lengths)*100:>5.1f}%)")
print(f"  >100 statements:  {over_100:>6,} ({over_100/len(lengths)*100:>5.1f}%)")
print(f"  >200 statements:  {over_200:>6,} ({over_200/len(lengths)*100:>5.1f}%)")

## Section 9: Vulnerable Lines per Function


In [ ]:
print("\nTrain split - vulnerable statements per vulnerable function:")
vuln_counts = [sum(r["label"]) for r in train if sum(r["label"]) > 0]

if vuln_counts:
    print(f"  Min:    {min(vuln_counts):>6}")
    print(f"  Max:    {max(vuln_counts):>6}")
    print(f"  Mean:   {sum(vuln_counts)/len(vuln_counts):>6.1f}")
    print(f"  Median: {sorted(vuln_counts)[len(vuln_counts)//2]:>6}")
else:
    print("  No vulnerable functions found")